# Notebook 05: Ray Core Fundamentals

## Why This Matters

Ray is the distributed computing framework of choice for ML workloads that go beyond a single machine. It powers Anyscale, is used by OpenAI (early GPT training infrastructure), Shopify, Instacart, and hundreds of other organizations. Where PyTorch DDP handles multi-GPU within a training job, Ray handles orchestrating many jobs across many machines -- hyperparameter search, data preprocessing, model serving, and experiment management. Understanding Ray's core primitives (tasks, actors, object store) is the foundation for all its higher-level libraries.


In [ ]:
%pip install -q 'ray[default]' numpy matplotlib

In [ ]:
import ray
import numpy as np
import time
import matplotlib.pyplot as plt

# Initialize Ray (single-node for this notebook)
if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, ignore_reinit_error=True)

print(f"Ray version: {ray.__version__}")
print(f"Cluster resources: {ray.cluster_resources()}")
print("Ready.")

## 1. Ray Architecture

Ray consists of:
- **Head node**: GCS (Global Control Service), scheduler, object directory
- **Worker nodes**: run tasks and actors
- **Object store (Plasma)**: shared-memory store for zero-copy object passing between workers
- **Raylet**: per-node daemon that manages local scheduling and object store

The key design insight: **tasks and actors return futures (ObjectRef)**. Computation is asynchronous by default. You only block when you call `ray.get()`.

This is fundamentally different from Python's `multiprocessing`: Ray can schedule across machines, passes objects via zero-copy shared memory, and handles failures.


In [ ]:
# Ray tasks: stateless parallel functions

@ray.remote
def compute_pi_sample(n_samples):
    """Monte Carlo pi estimation: count points inside unit circle."""
    import numpy as np
    x = np.random.uniform(-1, 1, n_samples)
    y = np.random.uniform(-1, 1, n_samples)
    inside = np.sum(x**2 + y**2 <= 1)
    return inside, n_samples

# Sequential computation
t0 = time.perf_counter()
total_inside = 0
total_samples = 0
for _ in range(8):
    inside, n = compute_pi_sample.remote.__wrapped__(1_000_000)
    total_inside += inside
    total_samples += n
seq_time = time.perf_counter() - t0

# Parallel computation with Ray
t0 = time.perf_counter()
futures = [compute_pi_sample.remote(1_000_000) for _ in range(8)]
results = ray.get(futures)  # blocking: wait for all
total_inside_ray = sum(r[0] for r in results)
total_samples_ray = sum(r[1] for r in results)
ray_time = time.perf_counter() - t0

pi_estimate = 4 * total_inside_ray / total_samples_ray
print(f'Pi estimate: {pi_estimate:.6f} (true: {np.pi:.6f})')
print(f'Sequential:  {seq_time:.2f}s')
print(f'Ray parallel: {ray_time:.2f}s')
print(f'Speedup: {seq_time/ray_time:.2f}x')

## 2. Ray Actors: Stateful Distributed Objects

While tasks are stateless (run-and-forget), **actors** are stateful. An actor is a class where:
- Each method call is asynchronous
- State is maintained between calls
- The actor runs in a dedicated process (or on a remote node)

Actors are perfect for: parameter servers, model servers, stream processors, shared caches.


In [ ]:
# Ray Actor: stateful counter / parameter server

@ray.remote
class ParameterServer:
    def __init__(self, n_params):
        self.params = np.zeros(n_params, dtype=np.float32)
        self.step = 0
        self.lr = 0.01
    
    def get_params(self):
        return self.params.copy()
    
    def apply_gradient(self, grad):
        self.params -= self.lr * grad
        self.step += 1
        return self.step
    
    def get_step(self):
        return self.step

# Create an actor (runs in its own process)
ps = ParameterServer.remote(n_params=100)

# Multiple workers apply gradients asynchronously
@ray.remote
def compute_gradient(params, worker_id):
    # Simulate computing gradient
    np.random.seed(worker_id)
    return np.random.randn(len(params)).astype(np.float32) * 0.1

# Training loop: workers pull params, compute grads, push grads
for step in range(5):
    params = ray.get(ps.get_params.remote())
    # Launch 4 workers in parallel
    grad_futures = [compute_gradient.remote(params, w) for w in range(4)]
    grads = ray.get(grad_futures)
    mean_grad = np.mean(grads, axis=0)
    step_num = ray.get(ps.apply_gradient.remote(mean_grad))

final_params = ray.get(ps.get_params.remote())
final_step = ray.get(ps.get_step.remote())
print(f'Final step: {final_step}')
print(f'Params norm: {np.linalg.norm(final_params):.4f}')
print(f'Params[0:3]: {final_params[:3]}')

## 3. Object Store: Zero-Copy Sharing

Ray's **plasma object store** is a shared memory system. When you pass a large numpy array as an argument to a remote function:
1. It is serialized into the object store once
2. All workers read it via shared memory (zero-copy if numpy)
3. No serialization/deserialization between local workers

This is dramatically faster than Python's multiprocessing.Queue for large arrays.


In [ ]:
# Object store: passing large arrays efficiently

@ray.remote
def process_chunk(data_ref, start, end):
    # data_ref is a reference to an object in the object store
    data = data_ref  # Ray already passed this via zero-copy shared memory
    chunk = data[start:end]
    return float(chunk.sum())

# Create a large array and put it in the object store
large_array = np.random.randn(1_000_000).astype(np.float32)

# Without ray.put: array is copied for each remote function call
t0 = time.perf_counter()
futures = [process_chunk.remote(large_array, i * 100_000, (i+1) * 100_000)
           for i in range(10)]
results = ray.get(futures)
t_no_put = time.perf_counter() - t0

# With ray.put: array is stored once, all functions reference the same object
t0 = time.perf_counter()
array_ref = ray.put(large_array)
futures = [process_chunk.remote(array_ref, i * 100_000, (i+1) * 100_000)
           for i in range(10)]
results_put = ray.get(futures)
t_with_put = time.perf_counter() - t0

print(f'Without ray.put: {t_no_put:.3f}s')
print(f'With ray.put:    {t_with_put:.3f}s')
print(f'Results match: {np.isclose(sum(results), sum(results_put))}')
print(f'\nray.put stores the array ONCE in the object store.')
print(f'All workers read it via zero-copy shared memory.')

## 4. ray.wait: Non-Blocking Result Collection

`ray.get(futures)` blocks until ALL futures complete. `ray.wait(futures)` returns as soon as K futures complete (default K=1). This enables better resource utilization -- start new tasks as soon as slots free up.


In [ ]:
# ray.wait: collect results as they complete

@ray.remote
def slow_task(task_id, duration):
    time.sleep(duration)
    return task_id, duration

# 10 tasks with varying durations
np.random.seed(42)
durations = np.random.uniform(0.1, 1.0, 10)
futures = [slow_task.remote(i, d) for i, d in enumerate(durations)]

print('Collecting results as they complete (ray.wait):')
remaining = futures.copy()
completion_order = []

t0 = time.perf_counter()
while remaining:
    done, remaining = ray.wait(remaining, num_returns=1)
    task_id, duration = ray.get(done[0])
    elapsed = time.perf_counter() - t0
    completion_order.append(task_id)
    if len(completion_order) <= 5 or len(completion_order) == 10:
        print(f'  Task {task_id} (dur={duration:.2f}s) completed at t={elapsed:.2f}s')

print(f'\nCompletion order: {completion_order}')
print(f'Expected (approx shortest first): {np.argsort(durations).tolist()[:5]}...')

## 5. Hyperparameter Search with Ray Tasks

Demonstrating Ray's composability: parallel hyperparameter search using just `@ray.remote` tasks.


In [ ]:
# Parallel hyperparameter search with Ray tasks

@ray.remote
def train_model(lr, n_hidden, n_epochs=20, seed=0):
    import numpy as np
    np.random.seed(seed)
    # Synthetic 2D XOR problem
    N = 200
    X = np.random.randn(N, 2)
    y = ((X[:, 0] * X[:, 1]) > 0).astype(float)
    
    # Two-layer MLP (numpy only)
    W1 = np.random.randn(2, n_hidden) * np.sqrt(2/2)
    b1 = np.zeros(n_hidden)
    W2 = np.random.randn(n_hidden, 1) * np.sqrt(2/n_hidden)
    b2 = np.zeros(1)
    
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(X):
        h = np.maximum(0, X @ W1 + b1)
        return sigmoid(h @ W2 + b2).ravel()
    
    final_acc = 0.5
    for epoch in range(n_epochs):
        h = np.maximum(0, X @ W1 + b1)
        pred = sigmoid(h @ W2 + b2).ravel()
        error = pred - y
        
        dW2 = h.T @ error[:, None] / N
        db2 = error.mean()
        dh = (error[:, None] * W2.T) * (h > 0)
        dW1 = X.T @ dh / N
        db1 = dh.mean(axis=0)
        
        W2 -= lr * dW2; b2 -= lr * db2
        W1 -= lr * dW1; b1 -= lr * db1
        
        final_acc = np.mean((forward(X) > 0.5) == y)
    
    return {'lr': lr, 'n_hidden': n_hidden, 'accuracy': float(final_acc)}

# Grid search in parallel
lrs = [0.01, 0.05, 0.1, 0.5]
hidden_sizes = [4, 8, 16, 32]

t0 = time.perf_counter()
futures = [train_model.remote(lr, nh, seed=42)
           for lr in lrs for nh in hidden_sizes]
results = ray.get(futures)
elapsed = time.perf_counter() - t0

# Find best
results.sort(key=lambda x: x['accuracy'], reverse=True)
print(f'Ran {len(results)} experiments in {elapsed:.2f}s')
print('Top 5 configurations:')
for r in results[:5]:
    print(f'  lr={r["lr"]:.3f}, n_hidden={r["n_hidden"]:2d}: acc={r["accuracy"]:.3f}')

## Summary

### Key Concepts

| Concept | API | Use case |
|---------|-----|---------|
| Task | `@ray.remote` function | Stateless parallel computation |
| Actor | `@ray.remote` class | Stateful distributed object |
| Object store | `ray.put(arr)` | Zero-copy large array sharing |
| Get results | `ray.get(futures)` | Block until all done |
| Non-blocking | `ray.wait(futures, num_returns=k)` | Process results as they arrive |

**Ray vs alternatives**:
- vs `multiprocessing`: Ray works across machines, zero-copy object store, better failure handling
- vs `Dask`: Ray is better for heterogeneous tasks (tasks + actors + serving); Dask better for dataframe ops
- vs `Celery`: Ray has object store and actors; Celery is better for web task queues

**Next**: `06_ray_train.ipynb` -- distributed PyTorch training with Ray's DDP/FSDP abstraction.
